# Analyze Baselines

In [241]:
from pathlib import Path
import json

import pandas as pd

In [242]:
from temp import (
    mis_metrics,
    plot_mis_predictions,
    temp_metrics
)

## Load Original Flows

In [243]:
# Configuration
dataset = "aitv2"
scenario = "santos"

out_dir = Path(f"../reports/{dataset}/{scenario}/baselines/temp")
out_dir.mkdir(parents=True, exist_ok=True)

In [244]:
df = pd.read_csv(
    f"../data/interim/{dataset}/{scenario}/flows_labeled/all_flows_labeled.csv"
)

df = df.sort_values("start_time").reset_index(drop=True)
df['t_rel'] = df['start_time'] - df['start_time'].min()

In [245]:
phase_bounds = (
    df[df['phase'] > 0]
    .groupby('phase')['t_rel']
    .agg(['min', 'max'])
)

phase_start = phase_bounds['min'].to_dict()
phase_end   = phase_bounds['max'].to_dict()

In [246]:
phase_bounds

,min,max
phase,,
1,7.305590,198962.549623
2,299710.474974,300162.491487
3,300174.736570,300251.641905
4,300253.783838,302244.616236


## Temp Plots and Custom Metrics

In [247]:
models = [
    "multiclass",
    "ensemble",
]

file_paths = []

for model in models:
    folder = Path(f"../experiments/{dataset}/{scenario}/baselines/{model}/metrics/")
    file_paths.extend(list(folder.iterdir()))

print(len(file_paths))

108


In [248]:
results = []

for file_path in file_paths:

    with open(file_path) as f:
        metrics = json.load(f)

    experiment_name = file_path.stem

    if "multiclass" in experiment_name:
        model = "multiclass"
    elif "ensemble" in experiment_name:
        model = "ensemble"

    # --- Load misclassification info ---
    real_flow_indices = metrics["real_flow_indices"]
    mis_y_pred = metrics["y_pred"]
    mis_y_true = metrics["y_true"]

    mis_df = df.iloc[real_flow_indices].copy()
    mis_df["y_true"] = mis_y_true
    mis_df["y_pred"] = mis_y_pred

    # --- Compute violation categories ---
    wrong, hard, soft, plausible = mis_metrics(mis_df, phase_start)
    hard_rate, soft_rate, temp_score = temp_metrics(metrics["Macro F1"], wrong, hard, soft)

    # --- Collect everything ---
    results.append({
        "model": f"{experiment_name}",
        "accuracy": metrics["Accuracy"],
        "precision" : metrics["Macro Precision"],
        "recall" : metrics["Macro Recall"], 
        "f1": metrics["Macro F1"],
        "total_wrong": len(wrong),
        "hard_violations": len(hard),
        "soft_violations": len(soft),
        "hard_rate": hard_rate,
        "soft_rate": soft_rate,
        "temporal_score": temp_score
    })

    # Create plots
    # plot_mis_predictions(df, phase_bounds, plausible, soft, hard, total_wrong, soft_rate, hard_rate, experiment_name, out_dir, show_plot=False)

## Create Metrics File

In [249]:
results_df = pd.DataFrame(results)

In [250]:
results_df_sorted_f1 = results_df.sort_values("f1", ascending=False)

# Save metrics to file
results_df_sorted_f1.to_csv(
    f"{out_dir.parent}/metrics.csv",
    index=False
)

In [251]:
results_df_sorted_f1.head(10)

,model,accuracy,precision,recall,f1,total_wrong,hard_violations,soft_violations,hard_rate,soft_rate,temporal_score
75,ensemble_aug_w100_full,0.998355,0.763793,0.948513,0.826444,232,20,202,0.086207,0.870690,0.609203
30,multiclass_aug_w100_full,0.998079,0.735316,0.946220,0.798918,271,24,233,0.088561,0.859779,0.582681
59,ensemble_aug_w10_full,0.997385,0.728288,0.922909,0.786436,367,5,283,0.013624,0.771117,0.625400
51,multiclass_aug_w10_full,0.995308,0.695040,0.924839,0.763093,661,44,406,0.066566,0.614221,0.606966
67,ensemble_full_w100_full,0.925606,0.558506,0.890635,0.643421,10494,1150,9160,0.109586,0.872880,0.414052
47,multiclass_full_w100_full,0.921530,0.510400,0.906095,0.605844,11069,1735,9038,0.156744,0.816515,0.364169
8,multiclass_reduced_w100_full,0.918581,0.502651,0.882732,0.595732,11484,2026,9137,0.176419,0.795629,0.348397
57,ensemble_reduced_w100_full,0.922551,0.499106,0.908442,0.587340,10925,1713,8978,0.156796,0.821785,0.344585
105,ensemble_aug_w100_1000b1000a,0.968269,0.505087,0.947192,0.582602,4473,1420,2894,0.317460,0.646993,0.294473
56,ensemble_aug_w10_1000b1000a,0.969503,0.514185,0.976289,0.576099,4302,735,3252,0.170851,0.755927,0.339488


In [252]:
results_df_sorted_temp = results_df.sort_values("temporal_score", ascending=False)

results_df_sorted_temp.head(10)

,model,accuracy,precision,recall,f1,total_wrong,hard_violations,soft_violations,hard_rate,soft_rate,temporal_score
59,ensemble_aug_w10_full,0.997385,0.728288,0.922909,0.786436,367,5,283,0.013624,0.771117,0.625400
75,ensemble_aug_w100_full,0.998355,0.763793,0.948513,0.826444,232,20,202,0.086207,0.870690,0.609203
51,multiclass_aug_w10_full,0.995308,0.695040,0.924839,0.763093,661,44,406,0.066566,0.614221,0.606966
30,multiclass_aug_w100_full,0.998079,0.735316,0.946220,0.798918,271,24,233,0.088561,0.859779,0.582681
67,ensemble_full_w100_full,0.925606,0.558506,0.890635,0.643421,10494,1150,9160,0.109586,0.872880,0.414052
47,multiclass_full_w100_full,0.921530,0.510400,0.906095,0.605844,11069,1735,9038,0.156744,0.816515,0.364169
8,multiclass_reduced_w100_full,0.918581,0.502651,0.882732,0.595732,11484,2026,9137,0.176419,0.795629,0.348397
57,ensemble_reduced_w100_full,0.922551,0.499106,0.908442,0.587340,10925,1713,8978,0.156796,0.821785,0.344585
56,ensemble_aug_w10_1000b1000a,0.969503,0.514185,0.976289,0.576099,4302,735,3252,0.170851,0.755927,0.339488
105,ensemble_aug_w100_1000b1000a,0.968269,0.505087,0.947192,0.582602,4473,1420,2894,0.317460,0.646993,0.294473
